# Classificador de fraude por rastreador

Usa os três modelos treinados no notebook anterior (resultados/modelos_finais.pkl) para classificar o arquivo de auditoria de um rastreador como **FRAUDE** ou **NÃO FRAUDE**, com o risco e a confiança de cada modelo. Nenhum indício isolado declara fraude: um episódio só é fraude se reunir pelo menos dois tipos de indício ao mesmo tempo e o risco combinado passar de 50%.

Seções: análise de um arquivo, ranking de uma pasta e interface web.

## Preparação do ambiente

In [ ]:
import os, sys

# o pacote fraude/ é gravado na pasta de trabalho pelas células da seção "Código do pacote fraude"
os.makedirs("fraude", exist_ok=True)

if "google.colab" in sys.modules:
    !pip -q install numpy pandas "scikit-learn>=1.2" "xgboost>=1.7" "matplotlib>=3.5" joblib openpyxl "gradio>=4.0"
    from google.colab import drive
    drive.mount("/content/drive")

sys.path.insert(0, os.getcwd())


## Código do pacote fraude
As células abaixo gravam os módulos do pacote `fraude/` na pasta de trabalho, então o notebook roda sozinho, sem clonar nenhum repositório. Execute-as na ordem, antes do restante.

In [ ]:
%%writefile fraude/__init__.py
"""Detecção de fraude em rastreadores veiculares a partir de planilhas de auditoria."""


In [ ]:
%%writefile fraude/tratamento.py
"""Leitura e limpeza das planilhas de auditoria exportadas pela plataforma de rastreamento.

Datas e durações são convertidas para tipos temporais antes do one-hot-encoding,
para que data, hora e minuto não se percam. Também corrige problemas da exportação:
coordenadas com separador de milhar, acentuação corrompida, ';' dentro do endereço
e cabeçalhos com nomes repetidos.
"""
from pathlib import Path

import numpy as np
import pandas as pd

DATE_FMT = "%d/%m/%Y %H:%M"   # formato de 'Location date', ex.: 02/01/2024 00:19
COORD_DECIMALS = 7            # -129.051.470 -> -12.9051470


def corrigir_texto(s):
    """Corrige acentuação corrompida (ex.: 'NÃƒO' -> 'NÃO')."""
    if not isinstance(s, str):
        return s
    try:
        return s.encode("cp1252").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return s


def para_datetime(serie):
    """Texto 'dd/mm/aaaa HH:MM[:SS]' ou datetime do Excel -> datetime64."""
    if pd.api.types.is_datetime64_any_dtype(serie):
        return serie
    s = serie.astype(str).str.strip()
    dt = pd.to_datetime(s, format=DATE_FMT, errors="coerce")
    faltando = dt.isna()
    if faltando.any():   # tenta com segundos / outros formatos
        dt[faltando] = pd.to_datetime(s[faltando], dayfirst=True, errors="coerce")
    return dt


def para_segundos(serie):
    """'HH:MM:SS' -> segundos (float). Aceita também timedelta/time do Excel."""
    s = serie.astype(str).str.strip()
    return pd.to_timedelta(s, errors="coerce").dt.total_seconds()


def corrigir_coordenada(valor, casas=COORD_DECIMALS):
    """
    A exportação gravou as coordenadas com pontos de milhar:
    '-129.051.470' -> -12.9051470  |  '-384.322.470' -> -38.4322470.
    Valores já numéricos e válidos (|x| <= 180) são mantidos. 0 vira NaN.
    """
    if pd.isna(valor):
        return np.nan
    if isinstance(valor, (int, float)) and abs(valor) <= 180 and float(valor).is_integer() is False:
        return float(valor) if valor != 0 else np.nan
    s = str(valor).strip().replace(",", ".")
    neg = s.startswith("-")
    digitos = "".join(ch for ch in s if ch.isdigit())
    if not digitos or int(digitos) == 0:
        return np.nan
    if s.count(".") == 1 and len(s.split(".")[1]) == casas:   # já está correto (ex.: -12.9051470)
        v = float(s)
    else:
        v = int(digitos) / 10 ** casas
    v = -v if neg else v
    return v if abs(v) <= 180 else np.nan


def sim_nao(serie):
    m = {"SIM": 1, "NÃO": 0, "NAO": 0, "TRUE": 1, "FALSE": 0, "1": 1, "0": 0}
    return serie.astype(str).str.strip().str.upper().map(m)


EVENTO_INICIO = "ENTERING GSM JAMMER AREA"
EVENTO_FIM    = "LEAVING GSM JAMMER AREA"

def rotular_fraude(df):
    estado = pd.Series(np.nan, index=df.index)
    estado[df["Reason Code"] == EVENTO_INICIO] = 1
    estado[df["Reason Code"] == EVENTO_FIM] = 1      # o próprio evento de saída ainda está no jammer
    # depois do evento de saída, volta a 0
    proximo_normal = (df["Reason Code"] == EVENTO_FIM).shift(fill_value=False)
    estado[proximo_normal & estado.isna()] = 0
    estado.iloc[0] = 0 if pd.isna(estado.iloc[0]) else estado.iloc[0]
    return estado.ffill().astype(int)


CATEGORICAS_ONEHOT = ["Reason Code", "HDOP STATUS"]
DESCARTAR = ["Place", "Plate", "Ibutton", "Creation date", "Reception date"]

LINHAS_CORRIGIDAS = {}   # arquivo -> nº de linhas com ";" extra

def ler_bruto(fp):
    """
    Lê a planilha de auditoria linha a linha.
    Linhas com mais campos que o cabeçalho (ex.: endereço em 'Place' contendo ';')
    têm o excedente reunido de volta na ÚLTIMA coluna, em vez de quebrar a leitura.
    Linhas com campos a menos são completadas com vazio.
    """
    if fp.suffix.lower() in (".xlsx", ".xls"):
        return pd.read_excel(fp, dtype=str)
    raw = fp.read_bytes()
    texto = None
    for enc in ("utf-8-sig", "cp1252", "latin-1"):
        try:
            texto = raw.decode(enc)
            break
        except UnicodeDecodeError:
            continue
    linhas = [l for l in texto.splitlines() if l.strip()]
    cab = [c.strip() for c in linhas[0].split(";")]
    n = len(cab)
    dados, corrigidas = [], 0
    for l in linhas[1:]:
        campos = l.split(";")
        if len(campos) > n:
            campos = campos[:n - 1] + [";".join(campos[n - 1:])]
            corrigidas += 1
        elif len(campos) < n:
            campos += [""] * (n - len(campos))
        dados.append(campos)
    LINHAS_CORRIGIDAS[fp.name] = corrigidas
    df = pd.DataFrame(dados, columns=cab, dtype=object)
    return df.replace({"": np.nan})


def limpar_texto(v):
    """Strip + correção de acentos, segura para NaN e valores não-texto."""
    if isinstance(v, str):
        v = corrigir_texto(v).strip()
        return v if v else np.nan
    return v


COLUNAS_DUPLICADAS = {}   # arquivo -> nomes repetidos no cabeçalho

def nomes_unicos(colunas, arquivo=""):
    """
    Alguns arquivos têm cabeçalho com nomes repetidos (ex.: duas colunas 'Lat'
    ou colunas vazias). Com nomes repetidos, df['col'] devolve um DataFrame
    em vez de uma Series. Aqui cada repetição recebe sufixo: Lat, Lat_2, ...
    Colunas sem nome viram 'sem_nome', 'sem_nome_2', ...
    """
    vistos, saida, repetidos = {}, [], []
    for c in colunas:
        c = str(c).strip() or "sem_nome"
        if c.startswith("Unnamed"):
            c = "sem_nome"
        if c in vistos:
            vistos[c] += 1
            repetidos.append(c)
            saida.append(f"{c}_{vistos[c]}")
        else:
            vistos[c] = 1
            saida.append(c)
    if repetidos:
        COLUNAS_DUPLICADAS[arquivo] = sorted(set(repetidos))
    return saida


def tratar_arquivo(fp):
    df = ler_bruto(fp).dropna(how="all")
    df.columns = nomes_unicos(df.columns, fp.name)
    # Remove colunas totalmente vazias e sem nome (sobras de ';' no fim do cabeçalho)
    df = df.drop(columns=[c for c in df.columns
                          if c.startswith("sem_nome") and df[c].isna().all()])
    for i in range(df.shape[1]):                      # por posição: nunca devolve DataFrame
        if df.iloc[:, i].dtype == object:
            df.iloc[:, i] = df.iloc[:, i].map(limpar_texto)
    if "Location date" not in df.columns:
        raise ValueError(f"coluna 'Location date' ausente. Colunas: {list(df.columns)}")

    # --- Tempo: mantém data + hora + minuto -------------------------
    df["timestamp"] = para_datetime(df["Location date"])
    df = df.dropna(subset=["timestamp"]).sort_values("timestamp", kind="stable")
    df = df.drop_duplicates().reset_index(drop=True)
    df["Data"]   = df["timestamp"].dt.strftime("%Y-%m-%d")
    df["Hora"]   = df["timestamp"].dt.hour
    df["Minuto"] = df["timestamp"].dt.minute
    df["hora_decimal"] = df["Hora"] + df["Minuto"] / 60
    df["hora_sin"] = np.sin(2 * np.pi * df["hora_decimal"] / 24)   # codificação cíclica
    df["hora_cos"] = np.cos(2 * np.pi * df["hora_decimal"] / 24)
    df["dia_semana"] = df["timestamp"].dt.dayofweek
    for c in ["Delay", "Reception delay"]:
        if c in df:
            df[c.replace(" ", "_") + "_seg"] = para_segundos(df[c])
    df = df.drop(columns=[c for c in ["Delay", "Reception delay", "Location date"] if c in df])

    # --- Numéricas ---------------------------------------------------
    for c in ["Speed", "GPS Satellites", "GSM Signal", "HDOP",
              "External Voltage", "Internal Battery", "Hodometer"]:
        if c in df:
            df[c] = pd.to_numeric(df[c].astype(str).str.replace(",", "."), errors="coerce")
    df["Lat"] = df["Lat"].map(corrigir_coordenada)
    df["Lng"] = df["Lng"].map(corrigir_coordenada)

    # --- Binárias ----------------------------------------------------
    for c in ["Ignition", "Block Car"]:
        if c in df:
            df[c] = sim_nao(df[c])

    # --- Rótulo (antes do one-hot, usa o texto do Reason Code) -------
    df["Fraude"] = rotular_fraude(df)

    # --- One-hot só das categóricas reais ----------------------------
    prefixos = {"Reason Code": "motivo", "HDOP STATUS": "hdop_status"}
    cols_oh = [c for c in CATEGORICAS_ONEHOT if c in df]
    df = pd.get_dummies(df, columns=cols_oh, prefix=[prefixos[c] for c in cols_oh], dtype=int)

    rastreador = (str(df["Module Id"].dropna().iloc[0])
                  if "Module Id" in df and df["Module Id"].notna().any()
                  else fp.stem.replace("_auditoria", ""))
    df["rastreador"] = rastreador
    df = df.drop(columns=[c for c in DESCARTAR + ["Module Id"] if c in df])
    df["timestamp"] = df["timestamp"].dt.strftime("%Y-%m-%d %H:%M:%S")
    return df, rastreador


In [ ]:
%%writefile fraude/features.py
"""Variáveis por registro, agregação em janelas de tempo e decisão combinada.

Todas as variáveis são calculadas dentro de um único rastreador, em ordem temporal,
e comparam o rastreador com ele mesmo (tensão relativa à própria mediana, área
habitual de circulação etc.).
"""
import re

import numpy as np
import pandas as pd

# Parâmetros de domínio
TARGET = "Fraude"
VAZAMENTO_PATTERNS = ["JAMMER"]      # colunas que definem o rótulo nunca viram feature

PARAMS = dict(
    JANELA_MIN=60,             # cada amostra do modelo = 1 janela de 60 min de um rastreador
    ROTULO_DURACAO_MIN=30,     # jammer só é FRAUDE se persistir >= 30 min (0 = rótulo antigo)
    DURACAO_MIN_SUSPEITA=30,   # persistência mínima das condições de sinal (min)
    DELAY_ALTO_SEG=600,        # atraso de recepção anormal (no exemplo, p90 normal ~ 400 s)
    HDOP_ALTO=3.0,
    SATELITES_MIN=4,
    GSM_FRACO_DBM=-105,        # no exemplo, dentro do jammer o GSM fica em -111 dBm
    GAP_TRANSMISSAO_MIN=45,    # parado, o rastreador transmite a cada ~30 min
    TENSAO_EXT_MIN=6.0,        # abaixo disso: alimentação externa cortada (V)
    PARADO_KMH=3,
    GRADE_GRAUS=0.01,          # célula de ~1 km para definir a "área habitual"
    FREQ_LOCAL_RARO=0.01,      # local onde o rastreador passa < 1% do tempo = incomum
    SALTO_KM=1.0,              # deslocamento após um apagão de transmissão
    KMH_MOVIMENTO=5,           # acima disso o veículo está andando
    # Violação de alimentação:
    # tensão externa cortada + veículo andando com ignição DESLIGADA
    MIN_REG_MOV_SEM_IGN=3,     # nº mínimo de registros andando com ignição off e energia cortada
    # Rótulo: "jammer" | "energia" | "ambos"  (fraudes_confirmadas.csv tem prioridade)
    ROTULO_MODO="ambos",
)

# Famílias de Reason Code: cada firmware escreve o mesmo evento de um jeito
# (ex.: "ENTERING GSM JAMMER AREA" x "GMS JAMMING ACTIVATED").
FAMILIAS_EVENTO = {
    "evt_jammer_inicio": r"ENTERING.*JAMM|JAMMING ACTIVATED|JAMMER ON",
    "evt_jammer_fim":    r"LEAVING.*JAMM|JAMMING DISABLED|JAMMER OFF",
    # Lista "REASON CODES (FRAUDE)": CALAMP, CONCOX, MXT, NEWTEC, PV_MXT,
    # SERTEC_RST, ST3XX, ST4XX, QUECLINK (54 códigos, todos cobertos abaixo)
    "evt_falha_energia": (r"EXTERNAL POWER FAIL|(EXTERNAL|MAIN) POWER( SUPPLY)? DISCONNECT"
                          r"|DISCONNECTED WITH THE MAIN POWER|(^|_)POWER DISCONNECT|POWER SUPPLY DISCONNECTED"
                          r"|REMOVE MAIN POWER|EXTERNAL BATTERY REMOVED|POWER SUPPLY ERROR|MAIN POWER (LOST|OFF)"),
    "evt_energia_ok":    (r"EXTERNAL POWER OK|(EXTERNAL|MAIN) POWER( SUPPLY)? CONNECT"
                          r"|(?<!DIS)CONNECTED WITH THE MAIN POWER|(^|_)POWER CONNECT"
                          r"|EXTERNAL BATTERY CONNECTED|POWER RECONNECT|MAIN POWER (ON|OK)"),
    "evt_tensao_baixa":  (r"LOW BATTERY|BATTERY LOW|LOW BACKUP BATTERY|BATTERY VOLTAGE BELOW"
                          r"|BELOW PWR THRES|DUE TO LOW POWER|BATTERY ERROR"),
    "evt_bateria_backup_removida": (r"BACKUP BATTERY UNPLUGGED|INTERNAL BATTERY REMOVED"
                                    r"|DISCONNECTED WITH THE BACK-?UP BATTERY|(^|_)BATTERY DISCONNECT"),
    "evt_bateria_backup_ok": (r"BACKUP BATTERY PLUGGED|INTERNAL BATTERY CONNECTED"
                              r"|(?<!DIS)CONNECTED WITH THE BACK-?UP BATTERY|(^|_)BATTERY CONNECT"),
    "evt_antena_gps_removida": r"GPS ANTENNA (UNPLUGGED|DISCONNECT)|(^|_)DISCONNECTED GPS ANTENNA|GPS SIGNAL FAILURE",
    "evt_antena_gps_ok": r"GPS ANTENNA (PLUGGED|CONNECT)|RECONNECTED GPS ANTENNA",
    "evt_desligamento":  r"POWER OFF|SHUT ?DOWN|(^|_)DISCONNECT$",
    "evt_religamento":   r"POWER ON|WAKE UP|GPRS FIRST ATTACHED|REBOOT|RESTART",
    "evt_seguranca":     r"UNAUTHORIZED DRIVER|PASSWORD WRONG|SUSPICIOUS|TRUCK (DIS)?CONNECTED",
    "evt_reconstrucao":  r"RECONSTRUCTION|STORED|BUFFER|OFFLINE",
}

# Famílias que, sozinhas, já indicam adulteração física do equipamento
EVENTOS_CRITICOS = ["evt_falha_energia", "evt_bateria_backup_removida",
                    "evt_antena_gps_removida", "evt_seguranca"]


def relatorio_cobertura(colunas_motivo):
    """Mostra quais Reason Codes da base caem em cada família e quais ficaram de fora."""
    linhas = []
    for c in colunas_motivo:
        fams = [f for f, p in FAMILIAS_EVENTO.items() if re.search(p, c.upper())]
        linhas.append({"reason_code": c.replace("motivo_", ""), "familias": ", ".join(fams) or "-"})
    return pd.DataFrame(linhas)


def familias_evento(df):
    """Soma as colunas one-hot de Reason Code em famílias comparáveis entre firmwares."""
    motivos = [c for c in df.columns if c.startswith("motivo_")]
    out = {}
    for fam, padrao in FAMILIAS_EVENTO.items():
        cols = [c for c in motivos if re.search(padrao, c.upper())]
        out[fam] = (df[cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
                    if cols else pd.Series(0, index=df.index)).astype("int16")
    return pd.DataFrame(out, index=df.index)


def gsm_em_dbm(gsm):
    """Alguns firmwares mandam GSM em dBm (-113 a -51), outros em CSQ (0 a 31; 99 = desconhecido)."""
    g = gsm.where(gsm != 99)
    if (g.dropna() >= 0).mean() > 0.8:          # escala CSQ
        return -113 + 2 * g.clip(0, 31)
    return g


def _num(df, c):
    if c in df.columns:
        return pd.to_numeric(df[c], errors="coerce")
    return pd.Series(np.nan, index=df.index, dtype="float64")


def haversine_vec(lat1, lon1, lat2, lon2, R=6371.0):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = (np.sin((lat2 - lat1) / 2) ** 2
         + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2)
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))


def duracao_continua(cond, ts):
    """Minutos desde o início da sequência ininterrupta em que cond é verdadeira."""
    cond = pd.Series(cond, index=ts.index).fillna(False).astype(bool)
    run = (cond != cond.shift()).cumsum()
    inicio = ts.groupby(run).transform("min")
    return ((ts - inicio).dt.total_seconds() / 60).where(cond, 0.0).astype("float32")


def rotulo_persistente(fraude, ts, dur_min):
    """Fraude = estado de jammer que dura pelo menos dur_min minutos."""
    fr = fraude.fillna(0).astype(int)
    run = (fr != fr.shift()).cumsum()
    fim = ts.shift(-1).fillna(ts)
    ini = ts.groupby(run).transform("min")
    fim_run = fim.groupby(run).transform("max")
    dur = ((fim_run - ini).dt.total_seconds() / 60).where(fr == 1)
    inicio_ep = (fr == 1) & (fr.shift(fill_value=0) != 1)
    return ((fr == 1) & (dur >= dur_min)).astype("int8"), dur, inicio_ep


def features_registro(df, P=PARAMS):
    """Features por registro, calculadas DENTRO de um único rastreador, em ordem temporal."""
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp"]).sort_values("timestamp", kind="stable").reset_index(drop=True)
    ts = df["timestamp"]
    D = P["DURACAO_MIN_SUSPEITA"]
    delay, hdop, sat = _num(df, "Reception_delay_seg"), _num(df, "HDOP"), _num(df, "GPS Satellites")
    gsm, vext, bat = _num(df, "GSM Signal"), _num(df, "External Voltage"), _num(df, "Internal Battery")
    lat, lng, vel, ign = _num(df, "Lat"), _num(df, "Lng"), _num(df, "Speed"), _num(df, "Ignition")
    lat, lng = lat.where(lat != 0), lng.where(lng != 0)
    gsm = gsm_em_dbm(gsm)
    if bat.fillna(0).eq(0).all():               # rastreador sem bateria interna medida
        bat = pd.Series(np.nan, index=df.index)
    if sat.max() > 24:                          # alguns firmwares somam satélites visíveis de várias constelações
        sat = sat.where(sat > 0)
    fam = familias_evento(df)

    out = pd.DataFrame({"timestamp": ts})
    dt_min = ts.diff().dt.total_seconds() / 60
    out["dt_min"] = dt_min

    # Sinal: só a DURAÇÃO conta (um pico curto = interferência comum, não fraude)
    out["delay_alto"] = (delay > P["DELAY_ALTO_SEG"]).astype("int8")
    out["dur_delay_alto"] = duracao_continua(delay > P["DELAY_ALTO_SEG"], ts)
    out["dur_hdop_alto"] = duracao_continua(hdop >= P["HDOP_ALTO"], ts)
    out["dur_sem_gps"] = duracao_continua(sat < P["SATELITES_MIN"], ts)
    out["dur_gsm_fraco"] = duracao_continua(gsm <= P["GSM_FRACO_DBM"], ts)

    # Energia: tensão externa e bateria interna, relativas ao próprio rastreador
    # Estado da alimentação pelos EVENTOS (falha -> cortada, ok -> normal).
    # Usado quando o firmware não envia tensão externa naquele registro.
    est = pd.Series(np.nan, index=df.index)
    est[fam["evt_falha_energia"] > 0] = 1
    est[fam["evt_energia_ok"] > 0] = 0
    est.iloc[0] = 0 if pd.isna(est.iloc[0]) else est.iloc[0]
    est = est.ffill()
    # o estado por evento expira em 12 h sem confirmação (evita "corte eterno" se faltar o evento de volta)
    ult_evt = ts.where((fam["evt_falha_energia"] > 0) | (fam["evt_energia_ok"] > 0)).ffill()
    est = est.where((ts - ult_evt) <= pd.Timedelta(hours=12), 0)
    ext_cortada = (vext < P["TENSAO_EXT_MIN"]).where(vext.notna(), est == 1).astype(bool)
    out["ext_cortada"] = ext_cortada.astype("int8")
    out["dur_ext_cortada"] = duracao_continua(ext_cortada, ts)
    out["vext"] = vext
    out["tensao_rel"] = vext - vext.median()
    out["queda_tensao"] = (-vext.diff()).clip(lower=0)
    out["bat"] = bat
    out["bateria_rel"] = bat - bat.median()
    out["queda_bateria"] = (-bat.diff()).clip(lower=0)

    # Posição: deslocamento, salto após apagão e área habitual do próprio rastreador
    d = haversine_vec(lat.shift(), lng.shift(), lat, lng)
    out["dist_km"] = d
    out["salto_pos_gap_km"] = np.where(dt_min > P["GAP_TRANSMISSAO_MIN"], d, 0.0)
    g = P["GRADE_GRAUS"]
    ok = lat.notna() & lng.notna()
    celula = pd.Series("", index=df.index)
    celula[ok] = ((lat[ok] / g).round().astype(int).astype(str) + "_"
                  + (lng[ok] / g).round().astype(int).astype(str))
    out["freq_local"] = celula.map(celula[ok].value_counts(normalize=True)).where(ok).astype("float32")
    out["dist_mediana_km"] = haversine_vec(lat, lng, lat.median(), lng.median())
    out["parado"] = (vel < P["PARADO_KMH"]).astype("int8")
    out["speed"], out["ignicao"] = vel, ign
    out["hora_sin"], out["hora_cos"] = _num(df, "hora_sin"), _num(df, "hora_cos")

    # Indicadores compostos
    degradado = pd.concat([out["dur_delay_alto"], out["dur_hdop_alto"],
                           out["dur_gsm_fraco"], out["dur_sem_gps"]], axis=1).max(axis=1)
    local_suspeito = (out["freq_local"] < P["FREQ_LOCAL_RARO"]) | (out["salto_pos_gap_km"] > P["SALTO_KM"])
    out["sinal_degradado_persistente"] = (degradado >= D).astype("int8")
    out["regra_combinada"] = ((degradado >= D) & local_suspeito).astype("int8")
    out["regra_energia"] = (out["dur_ext_cortada"] >= D).astype("int8")

    # Violação de alimentação: energia externa cortada E veículo andando com ignição desligada.
    # Fisicamente incoerente: o carro roda 100 km/h "desligado" enquanto o rastreador
    # está na bateria de backup e só depois envia a rota (ROUTE'S RECONSTRUCTION).
    andando = vel > P["KMH_MOVIMENTO"]
    out["mov_sem_ignicao"] = (andando & (ign == 0)).astype("int8")
    out["mov_sem_ign_cortada"] = (andando & (ign == 0) & ext_cortada).astype("int8")
    for c in fam.columns:
        if not c.startswith("evt_jammer"):
            out[c] = fam[c].values

    # Episódios contínuos de energia cortada
    run_ext = (ext_cortada != ext_cortada.shift()).cumsum()
    n_mov_ep = out["mov_sem_ign_cortada"].groupby(run_ext).transform("sum")
    dur_ep_ext = out["dur_ext_cortada"].groupby(run_ext).transform("max")
    violacao = ext_cortada & ((n_mov_ep >= P["MIN_REG_MOV_SEM_IGN"]) | (dur_ep_ext >= D))
    out["regra_violacao_energia"] = violacao.astype("int8")

    # Adulteração física: antena GPS ou bateria de backup removida, evento de segurança,
    # ou falha de energia — só vira alarme se houver CONFIRMAÇÃO no comportamento
    # (energia cortada ou veículo andando com ignição desligada nas 2 h seguintes).
    critico = fam[[c for c in EVENTOS_CRITICOS if c in fam]].sum(axis=1) > 0
    confirma = (ext_cortada | (out["mov_sem_ignicao"] == 1)).astype(int)
    # olha 2 h para a frente: para cada registro, existe confirmação em [t, t + 2 h]?
    # (ts já está em ordem crescente; soma acumulada + busca binária)
    t_ns = ts.to_numpy("datetime64[ns]")
    fim = np.searchsorted(t_ns, t_ns + np.timedelta64(2, "h"), side="right")
    acum = np.concatenate([[0], np.cumsum(confirma.to_numpy())])
    confirma_2h = acum[fim] - acum[np.arange(len(t_ns))]
    out["regra_adulteracao"] = (critico & (confirma_2h > 0)).astype("int8")

    for c in df.columns:   # contagem de eventos brutos (sem os que definem o rótulo de jammer)
        if c.startswith("motivo_") and not re.search("JAMM", c.upper()):
            out["n_" + c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype("int8")

    # Rótulo de jammer recalculado aqui a partir das famílias (vale para todos os firmwares)
    estado = pd.Series(np.nan, index=df.index)
    estado[fam["evt_jammer_inicio"] > 0] = 1
    estado[fam["evt_jammer_fim"] > 0] = 0
    estado.iloc[0] = 0 if pd.isna(estado.iloc[0]) else estado.iloc[0]
    jam_estado = estado.ffill().astype(int)
    jam_estado[fam["evt_jammer_fim"] > 0] = 1       # o registro de saída ainda está no jammer
    rot_jam, out["_dur_jammer"], out["_ini_jammer"] = rotulo_persistente(
        jam_estado, ts, P["ROTULO_DURACAO_MIN"])

    modo = P.get("ROTULO_MODO", "ambos")
    rot_en = out["regra_violacao_energia"]
    out[TARGET] = {"jammer": rot_jam, "energia": rot_en}.get(
        modo, ((rot_jam == 1) | (rot_en == 1)).astype("int8"))
    out["_rot_jammer"], out["_rot_energia"] = rot_jam, rot_en
    out["rastreador"] = str(df["rastreador"].iloc[0]) if "rastreador" in df.columns else "desconhecido"
    out["lat"], out["lng"] = lat, lng
    return out


def agregar_janelas(r, P=PARAMS):
    """Resume os registros em janelas de P['JANELA_MIN'] minutos: 1 linha = 1 janela."""
    r = r.copy()
    r["janela"] = r["timestamp"].dt.floor(f"{P['JANELA_MIN']}min")
    g = r.groupby("janela")
    J = pd.DataFrame({
        "n_registros": g.size(),
        "maior_gap_min": g["dt_min"].max(),
        "frac_delay_alto": g["delay_alto"].mean(),
        "dur_max_delay_alto": g["dur_delay_alto"].max(),
        "dur_max_hdop_alto": g["dur_hdop_alto"].max(),
        "dur_max_sem_gps": g["dur_sem_gps"].max(),
        "dur_max_gsm_fraco": g["dur_gsm_fraco"].max(),
        "frac_ext_cortada": g["ext_cortada"].mean(),
        "dur_max_ext_cortada": g["dur_ext_cortada"].max(),
        "tensao_rel_min": g["tensao_rel"].min(),
        "tensao_rel_media": g["tensao_rel"].mean(),
        "tensao_std": g["vext"].std(),
        "queda_tensao_max": g["queda_tensao"].max(),
        "bateria_rel_min": g["bateria_rel"].min(),
        "bateria_std": g["bat"].std(),
        "queda_bateria_max": g["queda_bateria"].max(),
        "dist_total_km": g["dist_km"].sum(),
        "salto_max_km": g["salto_pos_gap_km"].max(),
        "freq_local_min": g["freq_local"].min(),
        "dist_mediana_max_km": g["dist_mediana_km"].max(),
        "frac_parado": g["parado"].mean(),
        "frac_ignicao": g["ignicao"].mean(),
        "vel_media": g["speed"].mean(),
        "vel_max": g["speed"].max(),
        "hora_sin": g["hora_sin"].mean(),
        "hora_cos": g["hora_cos"].mean(),
        "sinal_degradado_persistente": g["sinal_degradado_persistente"].max(),
        "regra_combinada": g["regra_combinada"].max(),
        "regra_energia": g["regra_energia"].max(),
        "regra_violacao_energia": g["regra_violacao_energia"].max(),
        "n_mov_sem_ignicao": g["mov_sem_ignicao"].sum(),
        "n_mov_sem_ign_cortada": g["mov_sem_ign_cortada"].sum(),
        "n_evt_falha_energia": g["evt_falha_energia"].sum(),
        "n_evt_energia_ok": g["evt_energia_ok"].sum(),
        "n_evt_desligamento": g["evt_desligamento"].sum(),
        "n_evt_religamento": g["evt_religamento"].sum(),
        "n_evt_reconstrucao": g["evt_reconstrucao"].sum(),
        "n_evt_tensao_baixa": g["evt_tensao_baixa"].sum(),
        "n_evt_bateria_backup_removida": g["evt_bateria_backup_removida"].sum(),
        "n_evt_bateria_backup_ok": g["evt_bateria_backup_ok"].sum(),
        "n_evt_antena_gps_removida": g["evt_antena_gps_removida"].sum(),
        "n_evt_antena_gps_ok": g["evt_antena_gps_ok"].sum(),
        "n_evt_seguranca": g["evt_seguranca"].sum(),
        "regra_adulteracao": g["regra_adulteracao"].max(),
        "_rot_jammer": g["_rot_jammer"].max(),
        "_rot_energia": g["_rot_energia"].max(),
        "lat": g["lat"].median(),
        "lng": g["lng"].median(),
        TARGET: g[TARGET].max(),
        "_dur_jammer_max": g["_dur_jammer"].max(),
    })
    n_cols = [c for c in r.columns if c.startswith("n_motivo_")]
    if n_cols:
        J = J.join(g[n_cols].sum())
    J["n_registros_rel"] = J["n_registros"] / J["n_registros"].median()
    J = J.reset_index().rename(columns={"janela": "inicio_janela"})
    J["rastreador"] = r["rastreador"].iloc[0]
    return J


NAO_FEATURES = {"inicio_janela", "rastreador", "arquivo", "lat", "lng", TARGET}

# Variáveis que definem o rótulo de energia ficam fora do treino para o modelo
# não apenas reaprender a regra de rotulação.
DEFINEM_ROTULO_ENERGIA = {"regra_violacao_energia", "n_mov_sem_ign_cortada"}

def lista_features(J, P=PARAMS):
    excl = set(NAO_FEATURES)
    if P.get("ROTULO_MODO", "ambos") in ("energia", "ambos"):
        excl |= DEFINEM_ROTULO_ENERGIA
    return [c for c in J.columns
            if c not in excl and not c.startswith("_")
            and not any(p in c.upper() for p in VAZAMENTO_PATTERNS)]


# Decisão combinada (veredito do arquivo)
# Cada indício isolado NUNCA basta para declarar fraude. Um episódio só é fraude se
# reunir pelo menos MIN_EVIDENCIAS categorias diferentes de indício E a confiança
# combinada (modelo + indícios) atingir RISCO_MIN.
DECISAO = dict(
    RISCO_MIN=0.50,              # confiança mínima para declarar FRAUDE
    MIN_EVIDENCIAS=2,            # nº mínimo de categorias diferentes no mesmo episódio
    TOLERANCIA_JANELAS=1,        # janelas sem indício permitidas dentro de um episódio
    GAP_APAGAO_MIN=180,          # apagão de transmissão (min)
    # peso dos indícios combinados (nº de categorias -> probabilidade "de regra")
    PESO_EVIDENCIAS={2: 0.30, 3: 0.45, 4: 0.80},   # 2–3 indícios precisam do modelo; 4+ bastam
    # os indícios precisam ocorrer JUNTOS: na mesma janela ou nas vizinhas (±1 janela).
    # Indícios espalhados em dias diferentes NÃO se somam.
)

NOMES_EVIDENCIA = {
    "ev_energia": "energia externa cortada / falha ou tensão baixa",
    "ev_movimento_incoerente": "veículo andando com ignição desligada",
    "ev_desligamento": "equipamento desligado / religado",
    "ev_adulteracao": "antena GPS ou bateria de backup removida / evento de segurança",
    "ev_sinal_persistente": "sinal degradado por 30 min ou mais",
    "ev_jammer_persistente": "jammer por 30 min ou mais",
    "ev_posicao": "local incomum ou salto de posição após apagão",
    "ev_apagao": "apagão longo de transmissão",
}


def _col(J, c, padrao=0):
    return J[c].fillna(padrao) if c in J.columns else pd.Series(padrao, index=J.index)


def evidencias_janela(J, P=PARAMS, D=DECISAO):
    """Uma coluna booleana por CATEGORIA de indício, para cada janela."""
    return pd.DataFrame({
        # >= 2 leituras de tensão cortada: 1 leitura zerada isolada costuma ser falha de leitura
        "ev_energia": (_col(J, "frac_ext_cortada") * _col(J, "n_registros") >= 2)
                      | (_col(J, "n_evt_falha_energia") > 0)
                      | (_col(J, "n_evt_tensao_baixa") > 0),
        "ev_movimento_incoerente": _col(J, "n_mov_sem_ignicao") >= P["MIN_REG_MOV_SEM_IGN"],
        "ev_desligamento": (_col(J, "n_evt_desligamento") > 0) | (_col(J, "n_evt_religamento") > 0),
        "ev_adulteracao": (_col(J, "n_evt_antena_gps_removida") + _col(J, "n_evt_bateria_backup_removida")
                           + _col(J, "n_evt_seguranca")) > 0,
        # parado na garagem sem GPS é normal: sinal só conta com o veículo em movimento
        "ev_sinal_persistente": (_col(J, "sinal_degradado_persistente") > 0)
                                & (_col(J, "vel_max") > P["KMH_MOVIMENTO"]),
        "ev_jammer_persistente": _col(J, "_rot_jammer") > 0,
        "ev_posicao": (_col(J, "freq_local_min", 1.0) < P["FREQ_LOCAL_RARO"])
                      | (_col(J, "salto_max_km") > P["SALTO_KM"]),
        "ev_apagao": _col(J, "maior_gap_min") > D["GAP_APAGAO_MIN"],
    }, index=J.index)


def decisao_combinada(J, prob, P=PARAMS, D=DECISAO):
    """
    Agrupa janelas com indícios em episódios e decide cada episódio.
    confiança = 1 - (1 - p_modelo) * (1 - p_indicios)   se n_categorias >= MIN_EVIDENCIAS
              = no máximo RISCO_MIN - 1%                 se houver só 1 categoria (nunca é fraude)
    Retorna (veredito, confiança_fraude, tabela_de_episódios, evidências por janela).
    """
    J = J.reset_index(drop=True)
    prob = np.nan_to_num(np.asarray(prob, dtype=float))
    ev = evidencias_janela(J, P, D)
    tem = ev.any(axis=1).to_numpy()
    passo = pd.Timedelta(minutes=P["JANELA_MIN"])
    episodio = np.full(len(J), -1)
    idx = np.where(tem)[0]
    if len(idx):
        t = J["inicio_janela"].iloc[idx]
        episodio[idx] = (t.diff() > passo * (1 + D["TOLERANCIA_JANELAS"])).cumsum().to_numpy()
    # co-ocorrência local: categorias da janela + vizinhas imediatas
    t_all = J["inicio_janela"]
    ant_ok = ((t_all - t_all.shift(1)) == passo).to_numpy()
    prox_ok = ((t_all.shift(-1) - t_all) == passo).to_numpy()
    E = ev.to_numpy(dtype=bool)
    loc = E.copy()
    loc[1:] |= E[:-1] & ant_ok[1:, None]
    loc[:-1] |= E[1:] & prox_ok[:-1, None]
    n_loc = loc.sum(axis=1)
    teto = D["RISCO_MIN"] - 0.01
    linhas = []
    for e in np.unique(episodio[episodio >= 0]):
        m = episodio == e
        i_best = np.where(m)[0][np.argmax(n_loc[m])]
        cats = [c for j, c in enumerate(ev.columns) if loc[i_best, j]]
        n = len(cats)
        p_mod = float(prob[m].max())
        p_ind = D["PESO_EVIDENCIAS"].get(min(n, max(D["PESO_EVIDENCIAS"])), 0.0) if n >= 2 else 0.0
        conf = 1 - (1 - p_mod) * (1 - p_ind) if n >= D["MIN_EVIDENCIAS"] else min(p_mod, teto)
        ini = J.loc[m, "inicio_janela"].min()
        fim = J.loc[m, "inicio_janela"].max() + passo
        linhas.append(dict(inicio=ini, fim=fim, duracao_h=(fim - ini).total_seconds() / 3600,
                           n_indicios=n, indicios="; ".join(NOMES_EVIDENCIA[c] for c in cats),
                           prob_modelo=round(p_mod, 3), confianca=round(conf, 3),
                           fraude=conf >= D["RISCO_MIN"],
                           lat=J.loc[m, "lat"].median(), lng=J.loc[m, "lng"].median()))
    ep = pd.DataFrame(linhas, columns=["inicio", "fim", "duracao_h", "n_indicios", "indicios",
                                       "prob_modelo", "confianca", "fraude", "lat", "lng"])
    if len(ep):
        conf_arquivo = float(ep["confianca"].max())
    else:
        conf_arquivo = float(min(prob.max() if len(prob) else 0.0, teto))
    veredito = "FRAUDE" if conf_arquivo >= D["RISCO_MIN"] else "NÃO FRAUDE"
    ep = ep.sort_values("confianca", ascending=False).reset_index(drop=True)
    return veredito, conf_arquivo, ep, ev


In [ ]:
%%writefile fraude/figuras.py
"""Figuras usadas na monografia e na apresentação (PNG, 300 dpi)."""
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.metrics import auc, average_precision_score, precision_recall_curve, roc_curve

CORES = {"Regressão Logística": "#1f77b4", "Rede Neural (MLP)": "#2ca02c", "XGBoost": "#d62728"}
ROTULOS = {"precisao": "Precisão", "recall": "Recall", "f1": "F1",
           "pr_auc": "PR-AUC", "roc_auc": "ROC-AUC"}


def estilo():
    plt.rcParams.update({
        "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
        "font.size": 11, "axes.titlesize": 12,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.alpha": 0.3,
    })


def _fechar(fig, caminho):
    if caminho:
        Path(caminho).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(caminho)
    plt.show()
    plt.close(fig)


def matriz_confusao(c, titulo, caminho=None, classes=("Normal", "Fraude")):
    m = np.array([[c["VN"], c["FP"]], [c["FN"], c["VP"]]], dtype=float)
    fig, ax = plt.subplots(figsize=(4.4, 3.9))
    ax.imshow(m, cmap="Blues")
    ax.grid(False)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{int(m[i, j]):,}".replace(",", "."), ha="center", va="center",
                    fontsize=13, color="white" if m[i, j] > m.max() / 2 else "black")
    ax.set_xticks([0, 1], [f"Previsto\n{k}" for k in classes])
    ax.set_yticks([0, 1], [f"Real\n{k}" for k in classes])
    ax.set_title(titulo)
    _fechar(fig, caminho)


def curva_roc(y, probs, titulo, caminho=None):
    fig, ax = plt.subplots(figsize=(5.2, 4.6))
    for nome, p in probs.items():
        fpr, tpr, _ = roc_curve(y, p)
        ax.plot(fpr, tpr, color=CORES.get(nome), lw=1.8, label=f"{nome} (AUC = {auc(fpr, tpr):.3f})")
    ax.plot([0, 1], [0, 1], ls="--", color="gray", lw=1, label="Aleatório")
    ax.set_xlabel("Taxa de falsos positivos")
    ax.set_ylabel("Taxa de verdadeiros positivos")
    ax.set_title(titulo)
    ax.legend(loc="lower right", fontsize=9)
    _fechar(fig, caminho)


def curva_pr(y, probs, titulo, caminho=None):
    fig, ax = plt.subplots(figsize=(5.2, 4.6))
    for nome, p in probs.items():
        prec, rec, _ = precision_recall_curve(y, p)
        ap = average_precision_score(y, p)
        ax.plot(rec, prec, color=CORES.get(nome), lw=1.8, label=f"{nome} (PR-AUC = {ap:.3f})")
    ax.axhline(np.mean(y), ls="--", color="gray", lw=1, label=f"Aleatório ({np.mean(y):.3f})")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precisão")
    ax.set_ylim(0, 1.02)
    ax.set_title(titulo)
    ax.legend(loc="upper right", fontsize=9)
    _fechar(fig, caminho)


def barras_metricas(media, titulo, caminho=None, desvio=None,
                    metricas=("precisao", "recall", "f1", "pr_auc")):
    metricas = list(metricas)
    nomes = list(media.index)
    x = np.arange(len(metricas))
    largura = 0.8 / len(nomes)
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    for k, nome in enumerate(nomes):
        valores = media.loc[nome, metricas].astype(float).to_numpy()
        erro = desvio.loc[nome, metricas].astype(float).to_numpy() if desvio is not None else None
        barras = ax.bar(x + (k - (len(nomes) - 1) / 2) * largura, valores, largura,
                        yerr=erro, capsize=3, color=CORES.get(nome), label=nome)
        ax.bar_label(barras, fmt="%.2f", fontsize=8, padding=2)
    ax.set_xticks(x, [ROTULOS[m] for m in metricas])
    ax.set_ylim(0, 1.15)
    ax.set_title(titulo)
    ax.legend(fontsize=9, ncol=len(nomes), loc="upper center")
    _fechar(fig, caminho)


def boxplot_folds(resultados, metrica, titulo, caminho=None):
    nomes = list(dict.fromkeys(resultados["modelo"]))
    dados = [resultados.loc[resultados["modelo"] == n, metrica].dropna().to_numpy() for n in nomes]
    fig, ax = plt.subplots(figsize=(6.2, 4.2))
    caixas = ax.boxplot(dados, patch_artist=True, widths=0.5)
    for caixa, nome in zip(caixas["boxes"], nomes):
        caixa.set_facecolor(CORES.get(nome))
        caixa.set_alpha(0.35)
    rng = np.random.default_rng(0)
    for i, (d, nome) in enumerate(zip(dados, nomes), 1):
        ax.scatter(i + rng.uniform(-0.08, 0.08, len(d)), d, s=18, color=CORES.get(nome), zorder=3)
    ax.set_xticks(range(1, len(nomes) + 1), nomes)
    ax.set_ylabel(ROTULOS.get(metrica, metrica))
    ax.set_title(titulo)
    _fechar(fig, caminho)


def composicao_folds(comp, caminho=None):
    c = comp[comp["rep"] == 0]
    fig, ax = plt.subplots(figsize=(6.5, 4))
    x = np.arange(len(c))
    ax.bar(x, c["janelas_fraude"], color="#7f7f7f")
    for xi, (n, conc) in enumerate(zip(c["com_fraude"], c["concentracao"])):
        ax.text(xi, c["janelas_fraude"].iloc[xi], f"{n} rastr.\n{conc:.0%} em 1",
                ha="center", va="bottom", fontsize=8)
    ax.set_xticks(x, [f"Fold {f + 1}" for f in c["fold"]])
    ax.set_ylabel("Janelas com fraude no teste")
    ax.set_title("Distribuição das fraudes entre os folds")
    _fechar(fig, caminho)


def importancia(modelo, nome, features, X, y, caminho=None, n=15, seed=42):
    if nome == "Regressão Logística":
        v = pd.Series(modelo.named_steps["clf"].coef_[0], index=features)
        v = v.reindex(v.abs().sort_values().index).tail(n)
        eixo = "Coeficiente (variáveis padronizadas)"
    elif nome == "XGBoost":
        v = pd.Series(modelo.feature_importances_, index=features).sort_values().tail(n)
        eixo = "Importância (ganho)"
    else:
        # a MLP não tem coeficientes interpretáveis: usa importância por permutação
        rng = np.random.default_rng(seed)
        pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
        amostra = np.concatenate([rng.choice(pos, min(len(pos), 3000), replace=False),
                                  rng.choice(neg, min(len(neg), 17000), replace=False)])
        r = permutation_importance(modelo, X[amostra], y[amostra], scoring="average_precision",
                                   n_repeats=3, random_state=seed, n_jobs=1)
        v = pd.Series(r.importances_mean, index=features).sort_values().tail(n)
        eixo = "Redução média da PR-AUC (permutação)"

    fig, ax = plt.subplots(figsize=(7.2, 0.33 * len(v) + 1.3))
    ax.barh(v.index, v.to_numpy(), color=[CORES.get(nome) if x >= 0 else "#9e9e9e" for x in v])
    ax.set_xlabel(eixo)
    ax.set_title(f"{nome}: variáveis mais relevantes")
    ax.grid(axis="y", alpha=0)
    _fechar(fig, caminho)
    return v


In [ ]:
%%writefile fraude/analise.py
"""Classificação de um arquivo de auditoria com os modelos treinados."""
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from .features import agregar_janelas, decisao_combinada, features_registro
from .figuras import CORES
from .tratamento import tratar_arquivo


def carregar(caminho):
    """Aceita a planilha bruta de auditoria (.csv/.xlsx) ou um *_treated.csv."""
    fp = Path(caminho)
    if fp.name.endswith("_treated.csv"):
        return pd.read_csv(fp, sep=";", low_memory=False)
    df, _ = tratar_arquivo(fp)
    return df


def matriz(janelas, features):
    X = janelas.reindex(columns=features)
    contagens = [c for c in features if c.startswith("n_motivo_")]
    X[contagens] = X[contagens].fillna(0)
    X = X.to_numpy(dtype=np.float32)
    X[~np.isfinite(X)] = np.nan
    return X


def analisar_arquivo(caminho, pacote, modelos=None):
    """Classifica o arquivo de um rastreador com cada modelo pedido.

    Retorna (informações do arquivo, resultado por modelo, tabela de janelas).
    """
    params, decisao = pacote["params"], pacote["decisao"]
    janelas = agregar_janelas(features_registro(carregar(caminho), params), params)
    janelas = janelas.reset_index(drop=True)
    X = matriz(janelas, pacote["features"])

    resultados = {}
    for nome in modelos or list(pacote["modelos"]):
        prob = pacote["modelos"][nome]["modelo"].predict_proba(X)[:, 1]
        veredito, risco, episodios, _ = decisao_combinada(janelas, prob, params, decisao)
        confianca = risco if veredito == "FRAUDE" else 1 - risco
        resultados[nome] = dict(
            veredito=veredito,
            risco_pct=round(100 * risco, 1),
            confianca_pct=round(100 * confianca, 1),
            episodios_fraude=int(episodios["fraude"].sum()) if len(episodios) else 0,
            episodios_com_indicios=len(episodios),
            episodios=episodios,
            prob=prob,
        )

    votos = sum(r["veredito"] == "FRAUDE" for r in resultados.values())
    info = dict(
        arquivo=Path(caminho).name,
        rastreador=str(janelas["rastreador"].iloc[0]),
        periodo=f"{janelas['inicio_janela'].min():%d/%m/%Y} a {janelas['inicio_janela'].max():%d/%m/%Y}",
        janelas=len(janelas),
        consenso="FRAUDE" if votos > len(resultados) / 2 else "NÃO FRAUDE",
        votos_fraude=f"{votos} de {len(resultados)}",
    )
    return info, resultados, janelas


def tabela_resumo(resultados):
    return pd.DataFrame([
        {"Modelo": nome, "Veredito": r["veredito"], "Risco (%)": r["risco_pct"],
         "Confiança (%)": r["confianca_pct"], "Episódios de fraude": r["episodios_fraude"],
         "Episódios com indícios": r["episodios_com_indicios"]}
        for nome, r in resultados.items()
    ])


def tabela_episodios(resultados, so_fraude=False):
    partes = []
    for nome, r in resultados.items():
        ep = r["episodios"]
        if len(ep):
            partes.append(ep.assign(modelo=nome))
    if not partes:
        return pd.DataFrame(columns=["modelo", "inicio", "fim", "duracao_h", "n_indicios",
                                     "indicios", "confianca", "fraude"])
    ep = pd.concat(partes, ignore_index=True)
    if so_fraude:
        ep = ep[ep["fraude"]]
    ep = ep.sort_values(["fraude", "confianca"], ascending=False)
    ep["inicio"] = pd.to_datetime(ep["inicio"]).dt.strftime("%d/%m/%Y %H:%M")
    ep["fim"] = pd.to_datetime(ep["fim"]).dt.strftime("%d/%m/%Y %H:%M")
    ep["duracao_h"] = ep["duracao_h"].astype(float).round(1)
    ep["confianca"] = (100 * ep["confianca"].astype(float)).round(1)
    ep["fraude"] = ep["fraude"].map({True: "sim", False: "não"})
    return ep[["modelo", "fraude", "confianca", "inicio", "fim", "duracao_h",
               "n_indicios", "indicios", "lat", "lng"]].reset_index(drop=True)


def grafico_risco(janelas, resultados, decisao, titulo="", caminho=None):
    """Risco ao longo do tempo: um painel por modelo.

    Faixas vermelhas = episódios classificados como fraude;
    faixas laranja = indícios que não chegaram ao limite.
    """
    nomes = list(resultados)
    fig, eixos = plt.subplots(len(nomes), 1, figsize=(13, 2.4 * len(nomes) + 0.8),
                              sharex=True, squeeze=False)
    for ax, nome in zip(eixos[:, 0], nomes):
        r = resultados[nome]
        ax.plot(janelas["inicio_janela"], r["prob"], lw=0.7, color=CORES.get(nome, "C0"))
        for _, e in r["episodios"].iterrows():
            ax.axvspan(e["inicio"], e["fim"], lw=0,
                       color="red" if e["fraude"] else "orange",
                       alpha=0.25 if e["fraude"] else 0.12)
        ax.axhline(decisao["RISCO_MIN"], color="red", ls="--", lw=0.8)
        ax.set_ylim(-0.02, 1.02)
        ax.set_ylabel("Risco")
        ax.set_title(f"{nome}: {r['veredito']} (risco {r['risco_pct']:.1f}%)", loc="left", fontsize=10)
    if titulo:
        fig.suptitle(titulo)
    fig.tight_layout()
    if caminho:
        Path(caminho).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(caminho, dpi=300, bbox_inches="tight")
    return fig


In [ ]:
%%writefile fraude/interface.py
"""Interface web (Gradio) para classificar o arquivo de um rastreador."""
import tempfile
from pathlib import Path

import gradio as gr

from .analise import analisar_arquivo, grafico_risco, tabela_episodios, tabela_resumo


def _cartao(info):
    fraude = info["consenso"] == "FRAUDE"
    cor = "#c62828" if fraude else "#2e7d32"
    return (
        f"<div style='padding:16px;border-radius:10px;border:2px solid {cor};"
        f"background:{cor}14;color:{cor};text-align:center'>"
        f"<div style='font-size:28px;font-weight:700'>{info['consenso']}</div>"
        f"<div style='font-size:15px'>Modelos que indicaram fraude: {info['votos_fraude']}</div></div>"
        f"<p><b>Rastreador:</b> {info['rastreador']} &nbsp;|&nbsp; <b>Período:</b> {info['periodo']}"
        f" &nbsp;|&nbsp; <b>Janelas analisadas:</b> {info['janelas']}</p>"
    )


def criar_interface(pacote):
    nomes = list(pacote["modelos"])

    def executar(caminho, escolha):
        if not caminho:
            return "<p>Envie um arquivo de auditoria.</p>", None, None, None, None
        modelos = nomes if escolha == "Todos" else [escolha]
        try:
            info, resultados, janelas = analisar_arquivo(caminho, pacote, modelos)
        except Exception as e:
            erro = f"<p style='color:#c62828'>Não foi possível analisar o arquivo: {type(e).__name__}: {e}</p>"
            return erro, None, None, None, None

        resumo = tabela_resumo(resultados)
        saida = Path(tempfile.gettempdir()) / f"{Path(caminho).stem}_resultado.csv"
        resumo.assign(arquivo=info["arquivo"], periodo=info["periodo"]).to_csv(saida, sep=";", index=False)
        figura = grafico_risco(janelas, resultados, pacote["decisao"])
        return _cartao(info), figura, resumo, tabela_episodios(resultados), str(saida)

    with gr.Blocks(title="Detecção de fraude em rastreadores") as demo:
        gr.Markdown("## Detecção de fraude em rastreadores veiculares\n"
                    "Envie a planilha de auditoria de **um** rastreador (CSV separado por ';' ou XLSX).")
        with gr.Row():
            arquivo = gr.File(label="Arquivo de auditoria", file_types=[".csv", ".xlsx", ".xls"],
                              type="filepath")
            modelo = gr.Dropdown(["Todos"] + nomes, value="Todos", label="Modelo")
        botao = gr.Button("Analisar", variant="primary")
        cartao = gr.HTML()
        grafico = gr.Plot(label="Risco ao longo do tempo")
        resumo = gr.Dataframe(label="Resultado por modelo")
        episodios = gr.Dataframe(label="Episódios com indícios", wrap=True)
        download = gr.File(label="Baixar resultado (CSV)")
        botao.click(executar, [arquivo, modelo], [cartao, grafico, resumo, episodios, download])
    return demo


In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd

from fraude import figuras as fig
from fraude.analise import analisar_arquivo, grafico_risco, tabela_episodios, tabela_resumo

DRIVE = Path("/content/drive/MyDrive")
DIR_BRUTOS = DRIVE / "dados_brutos"          # planilhas de auditoria originais
DIR_TRATADOS = DRIVE / "dataset_tratado"     # *_treated.csv
DIR_RESULT = DIR_TRATADOS / "resultados"
DIR_FIG = DIR_RESULT / "figuras"
DIR_FIG.mkdir(parents=True, exist_ok=True)

pacote = joblib.load(DIR_RESULT / "modelos_finais.pkl")
fig.estilo()
print("modelos:", ", ".join(pacote["modelos"]))

## 1. Análise de um arquivo

In [ ]:
ARQUIVO = DIR_BRUTOS / "69251992613cd7d158638cb9_auditoria.csv"   # troque pelo arquivo desejado

info, resultados, janelas = analisar_arquivo(ARQUIVO, pacote)
print(f"{info['arquivo']} | {info['periodo']} | consenso: {info['consenso']} ({info['votos_fraude']} modelos)")
display(tabela_resumo(resultados))
grafico_risco(janelas, resultados, pacote["decisao"], caminho=DIR_FIG / f"risco_{Path(ARQUIVO).stem}.png")
plt.show()
tabela_episodios(resultados, so_fraude=True).head(20)

### Enviar um arquivo do computador

In [ ]:
from google.colab import files

for nome in files.upload():
    info, resultados, janelas = analisar_arquivo(nome, pacote)
    print(f"{info['arquivo']}: {info['consenso']} ({info['votos_fraude']} modelos)")
    display(tabela_resumo(resultados))
    grafico_risco(janelas, resultados, pacote["decisao"])
    plt.show()

## 2. Interface web
Gera uma página para enviar o arquivo e receber o resultado. O link público vale por 72 horas; para uso permanente, rode a interface fora do Colab (pip install gradio e launch() em um script local).

In [ ]:
from fraude.interface import criar_interface

criar_interface(pacote).launch(share=True)